In [ ]:
import pandas as pd

number_of_groups = 5
data = pd.read_csv('TestResults/MathTestDataExample.csv')
column_names = data.columns
names = data[column_names[0]]
features = data[column_names[1:]]
data

,Student Name,1,2,3,4,5,6,7,8,9,10,11
0,Nuhamin,True,True,True,False,True,True,False,True,True,True,True
1,Dylan,True,True,True,False,True,True,True,True,True,True,True
2,Aylin,True,True,True,False,True,True,False,False,True,True,True
3,Tewoflos,True,True,True,False,False,True,True,False,False,False,True
4,Nasir,False,False,False,False,False,False,False,False,False,False,False
5,Fadil,True,True,True,False,False,True,True,True,True,True,True
6,Allison,True,True,True,False,False,True,True,False,True,True,True
7,Zahara,True,True,True,False,True,True,True,True,True,True,True
8,Ryan,True,False,True,False,False,True,True,True,True,True,True
9,Evan,True,True,True,True,True,False,False,True,True,True,True


In [43]:
from sklearn.metrics import pairwise_distances
import numpy as np

# Compute distance matrix using Hamming distance
distance_matrix = pairwise_distances(features, metric='hamming')

distance_df = pd.DataFrame(distance_matrix)
distance_df['Names'] = names

name_dictionary = {}
for i in range(len(names)):
    name_dictionary[i] = names[i]
print(name_dictionary)

out_d = data.melt(f'{column_names[0]}').query('value == False').groupby(f'{column_names[0]}')['variable'].agg(list).to_dict()


{0: 'Nuhamin', 1: 'Dylan', 2: 'Aylin', 3: 'Tewoflos', 4: 'Nasir', 5: 'Fadil', 6: 'Allison', 7: 'Zahara', 8: 'Ryan', 9: 'Evan', 10: 'Markan', 11: 'Jad', 12: 'Ahmed', 13: 'Franklin', 14: 'Sandro', 15: 'Naod', 16: 'Janna', 17: 'Emmanuel', 18: 'Brittany', 19: 'Dominic', 20: 'Zyaire', 21: 'Naole', 22: 'Rafael', 23: 'Deon', 24: 'Abreham'}


In [15]:
import numpy as np
import pulp

# Example: 10 data points and pairwise distances
n_points = features.shape[0]
n_groups = 5

# Problem definition
prob = pulp.LpProblem("Minimize_IntraGroup_Distances", pulp.LpMinimize)

# Decision variables
x = pulp.LpVariable.dicts("x", ((i, k) for i in range(n_points) for k in range(n_groups)), cat="Binary")
y = pulp.LpVariable.dicts("y", ((i, j) for i in range(n_points) for j in range(i+1, n_points)), cat="Binary")

# Objective: minimize total intra-group distances
prob += pulp.lpSum(distance_matrix[i][j] * y[i, j] for i in range(n_points) for j in range(i+1, n_points))

# Each point must belong to exactly one group
for i in range(n_points):
    prob += pulp.lpSum(x[i, k] for k in range(n_groups)) == 1

# Ensure y[i][j] = 1 if and only if i and j are in the same group
for i in range(n_points):
    for j in range(i+1, n_points):
        for k in range(n_groups):
            prob += y[i, j] >= x[i, k] + x[j, k] - 1

# Solve
# prob.solve(pulp.PULP_CBC_CMD(maxSeconds=180))
prob.solve(pulp.PULP_CBC_CMD(timeLimit = 300))

# Extract solution
groups = {k: [] for k in range(n_groups)}
for i in range(n_points):
    for k in range(n_groups):
        if pulp.value(x[i, k]) > 0.5:
            groups[k].append(i)

print("Groups:")
for k, members in groups.items():
    member_names = [name_dictionary[member] for member in members]
    print(f"Group {k}: {member_names}")


Welcome to the CBC MILP Solver 
Version: 2.10.3 
Build Date: Dec 15 2019 

command line - /Users/kassandrabequer/.pyenv/versions/myproject/lib/python3.12/site-packages/pulp/apis/../solverdir/cbc/osx/i64/cbc /var/folders/w_/m47sxr8s3vxdns_y8lbsyp9h0000gn/T/46172d16c1c64028a91caac51d6a64c5-pulp.mps -sec 300 -timeMode elapsed -branch -printingOptions all -solution /var/folders/w_/m47sxr8s3vxdns_y8lbsyp9h0000gn/T/46172d16c1c64028a91caac51d6a64c5-pulp.sol (default strategy 1)
At line 2 NAME          MODEL
At line 3 ROWS
At line 1530 COLUMNS
At line 7303 RHS
At line 8829 BOUNDS
At line 9255 ENDATA
Problem MODEL has 1525 rows, 425 columns and 4625 elements
Coin0008I MODEL read with 0 errors
seconds was changed from 1e+100 to 300
Option for timeMode changed from cpu to elapsed
Continuous objective value is 0 - 0.00 seconds
Cgl0004I processed model has 1510 rows, 422 columns (422 integer (422 of which binary)) and 4580 elements
Cbc0038I Initial state - 50 integers unsatisfied sum - 25
Cbc0038I 

In [44]:
print("Groups:")
for k, members in groups.items():
    member_names = [name_dictionary[member] for member in members]
    print(f"Group {k+1}: {member_names}")
    print('Questions Wrong')
    for i in members:
        print(f'Name: {names[i]}')
        print(f'Questions: {out_d.get(names[i])}')


Groups:
Group 1: ['Nasir', 'Franklin', 'Brittany', 'Rafael']
Questions Wrong
Name: Nasir
Questions: ['1', '2', '3', '4', '5', '6', '7', '8', '9', '10', '11']
Name: Franklin
Questions: ['2', '4', '5', '8', '9']
Name: Brittany
Questions: ['2', '4', '7', '8', '10']
Name: Rafael
Questions: ['2', '4', '5', '7', '9', '10', '11']
Group 2: ['Dylan', 'Evan', 'Naod', 'Zyaire', 'Naole']
Questions Wrong
Name: Dylan
Questions: ['4']
Name: Evan
Questions: ['6', '7']
Name: Naod
Questions: ['4', '7']
Name: Zyaire
Questions: ['3', '4', '5', '7']
Name: Naole
Questions: ['2', '4', '5', '7']
Group 3: ['Nuhamin', 'Fadil', 'Zahara', 'Ryan', 'Sandro', 'Dominic']
Questions Wrong
Name: Nuhamin
Questions: ['4', '7']
Name: Fadil
Questions: ['4', '5']
Name: Zahara
Questions: ['4']
Name: Ryan
Questions: ['2', '4', '5']
Name: Sandro
Questions: ['5']
Name: Dominic
Questions: ['4', '5']
Group 4: ['Aylin', 'Jad', 'Ahmed', 'Janna', 'Emmanuel']
Questions Wrong
Name: Aylin
Questions: ['4', '7', '8']
Name: Jad
Questions: 